# Opdracht 1: Voorspel herseninfarct (MedPredict)

**Team:** *AMK-KLAS B*

**Teamleden en Kaggle-gebruikersnamen:**

| Naam | Kaggle-gebruikersnaam |
|---|---|
| **Ahmed Bashabeeb 25097172** | ahmedbashabeeb |
| **Khaleel Al-Aqel 24053872** | ** | 
| **Mehmet Kilinc 25135007** | *khaleelalaqel* |



## 1. Inleiding

In deze opdracht ontwikkelen we voor MedPredict een machine learning-model dat voorspelt of een patiënt een herseninfarct (stroke) krijgt, op basis van gegevens. We werken hiervoor met een dataset van ruim 33.000 patiënten en leveren onze voorspellingen in via een Kaggle-competitie. Naast een zo goed mogelijk voorspellend model kijken we ook naar de medische context: een gemiste


## 2. Data

We gebruiken drie bestanden uit de Kaggle-competitie
["Predicting stroke 2026-27"](https://www.kaggle.com/competitions/predicting-stroke-2026-27):

- **train.csv** — 33.550 patiënten, inclusief doelvariabele `stroke` (0/1).
- **test.csv** — 8.388 patiënten, zonder `stroke`-kolom.
- **submission_example.csv** — voorbeeldformaat voor de Kaggle-inlevering (`id`, `stroke`).


In [1]:
# Gedeelde imports (door iedereen te gebruiken/aan te vullen)
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", None)
plt.rcParams["figure.figsize"] = (6, 4)

RANDOM_STATE = 42 


**pd.set_option** zorgt ervoor dat pandas alle kolommen toont wanneer je een dataframe print, in plaats van kolommen af te korten met ...
<br>

**plt.rcParams** stelt de standaardgrootte in van elke grafiek die we met matplotlib maken

<br>

### Data inladen

In [ ]:
train_df = pd.read_csv("train.csv")
test_df = pd.read_csv("test.csv")

## 3. Opdracht 1: Exploratieve Data Analyse (EDA)


### 3.1 Kolombeschrijving

| Kolom | Betekenis | Type |
|---|---|---|
| id | Uniek patiënt-identificatienummer | identifier |
| gender | Geslacht van de patiënt: "Male", "Female" of "Other" | categorisch |
| age | Leeftijd van de patiënt in jaren | numeriek |
| hypertension | 0 = geen hoge bloeddruk, 1 = wel hoge bloeddruk | categorisch (binair) |
| heart_disease | 0 = geen hartaandoening, 1 = wel een hartaandoening | categorisch (binair) |
| ever_married | Burgerlijke staat: "No" of "Yes" | categorisch |
| work_type | Type werk: "children", "Govt_job", "Never_worked", "Private", "Self-employed" | categorisch |
| Residence_type | Woonomgeving: "Rural" of "Urban" | categorisch |
| avg_glucose_level | Gemiddeld glucosegehalte in het bloed | numeriek |
| bmi | Body mass index | numeriek |
| smoking_status | Rookgedrag: "formerly smoked", "never smoked", "smokes", "Unknown" | categorisch |
| stroke | **Doelvariabele**: 1 = patiënt heeft een herseninfarct gehad, 0 = niet | categorisch (target) |




### 5.6 Decision Tree

Code in deze paragraaf: [Claude, 2026. Prompt 3: Decision Tree stroke prediction](https://claude.ai/share/741e7b67-5336-4153-965e-d840720b4f7d).

#### 5.6.1 Hoe werkt een beslisboom?
Een beslisboom is eigenlijk gewoon "Wie is het?". Je stelt steeds een ja/nee-vraag, bijvoorbeeld "is iemand ouder dan 67?", en afhankelijk van het antwoord ga je naar links of naar rechts. Dat doe je een paar keer, tot je onderaan uitkomt in een blad (*leaf*). In dat blad kijk je hoeveel mensen een stroke hadden, en dat is de voorspelde kans. De bovenste vraag heet de *root*, de vragen daaronder zijn *decision nodes*.

De boom kiest bij elke stap de vraag die de groep het beste splitst. Hij kijkt daarbij alleen naar die ene stap en niet verder vooruit, dat heet *greedy* (scikit-learn gebruikt hiervoor het CART-algoritme). Het fijne is dat je kunt lezen waarom hij iets zegt. Het nadeel is dat hij er heel anders uit kan zien als je de data een beetje verandert.

(Breiman et al., 1984; De Haagse Hogeschool, 2026a)

#### 5.6.2 Criteria voor het splitsen
Om te weten welke vraag het beste is, moet je meten hoe "rommelig" (*impure*) een groep is. Met $p_k$ = het aandeel van klasse $k$ in een groep:

$$Gini = 1 - \sum_k p_k^2 \qquad\qquad H = -\sum_k p_k \log_2 p_k$$

Zitten er alleen mensen zonder stroke in een groep, dan is die zuiver en zijn Gini en entropie allebei 0. Is het half om half, dan is het maximaal rommelig: Gini 0,5 en entropie 1. Bij een split reken je de onzuiverheid van links en rechts uit en neem je het gewogen gemiddelde, waarbij een grote groep zwaarder telt. Hoeveel dat lager is dan bij de groep ervoor, heet de *information gain*:

$$IG = I(\text{ouder}) - \Big(\tfrac{n_{links}}{n}\, I(\text{links}) + \tfrac{n_{rechts}}{n}\, I(\text{rechts})\Big)$$

De vraag met de grootste information gain wint. Hieronder rekenen we dat na op een klein groepje van 20 patiënten: 6 met en 14 zonder stroke.

(Breiman et al., 1984; James et al., 2021)

In [ ]:
# Prompt 3: Decision Tree stroke prediction
# Vragen als (feature, drempel), waarbij "ja" betekent: waarde <= drempel
SPLIT_QUESTIONS = [("age", 60), ("hypertension", 0.5)]
N_EXAMPLE_STROKE = 6
N_EXAMPLE_NO_STROKE = 14


def gini(y):
    """Berekent de Gini-impurity van een verzameling labels."""
    y = np.asarray(y, dtype=int)
    if len(y) == 0:
        return 0.0
    p = np.bincount(y, minlength=2) / len(y)
    return 1 - np.sum(p ** 2)


def entropy(y):
    """Berekent de entropie (in bits) van een verzameling labels."""
    y = np.asarray(y, dtype=int)
    if len(y) == 0:
        return 0.0
    p = np.bincount(y, minlength=2) / len(y)
    # Klassen met kans 0 tellen niet mee, want 0 * log(0) = 0
    p = p[p > 0]
    return np.sum(p * np.log2(1 / p))


def information_gain(y, mask, impurity=gini):
    """Berekent de information gain van een split met een booleaans masker."""
    y = np.asarray(y, dtype=int)
    mask = np.asarray(mask, dtype=bool)
    left, right = y[mask], y[~mask]
    # Gewogen gemiddelde impurity van de twee kinderen
    weighted = (len(left) * impurity(left)
                + len(right) * impurity(right)) / len(y)
    return impurity(y) - weighted


def draw_example(X, y, n_stroke, n_no_stroke):
    """Trekt een kleine steekproef met een vast aantal patiënten per klasse."""
    stroke_idx = y[y == 1].sample(n_stroke, random_state=RANDOM_STATE).index
    no_stroke_idx = y[y == 0].sample(
        n_no_stroke, random_state=RANDOM_STATE
    ).index
    example_idx = stroke_idx.append(no_stroke_idx)
    return X.loc[example_idx], y.loc[example_idx]


def describe_node(y, name):
    """Vat een knoop samen in aantallen, Gini en entropie."""
    y = np.asarray(y, dtype=int)
    return {
        "knoop": name,
        "aantal": len(y),
        "stroke": int(y.sum()),
        "geen stroke": int(len(y) - y.sum()),
        "gini": gini(y),
        "entropie": entropy(y),
    }


def explain_split(X, y, feature, threshold):
    """Laat stap voor stap zien hoe de information gain wordt berekend."""
    mask = (X[feature] <= threshold).to_numpy()
    y = np.asarray(y, dtype=int)
    nodes = pd.DataFrame([
        describe_node(y, "ouder"),
        describe_node(y[mask], f"ja: {feature} <= {threshold}"),
        describe_node(y[~mask], f"nee: {feature} > {threshold}"),
    ]).set_index("knoop")
    print(f"\nVraag: {feature} <= {threshold}?")
    display(nodes.round(3))
    gains = {}
    for column, impurity in [("gini", gini), ("entropie", entropy)]:
        parent, left, right = nodes[column]
        gain = information_gain(y, mask, impurity)
        # Ouder-impurity min het gewogen gemiddelde van de kinderen
        print(
            f"IG ({column}) = {parent:.3f} - ({mask.sum()}/{len(y)} * "
            f"{left:.3f} + {(~mask).sum()}/{len(y)} * {right:.3f}) "
            f"= {gain:.3f}"
        )
        gains[f"IG {column}"] = gain
    return gains


X_example, y_example = draw_example(
    X_train, y_train, N_EXAMPLE_STROKE, N_EXAMPLE_NO_STROKE
)
split_gains = {}
for feature, threshold in SPLIT_QUESTIONS:
    split_gains[f"{feature} <= {threshold}"] = explain_split(
        X_example, y_example, feature, threshold
    )
pd.DataFrame(split_gains).T.round(3)

**Uitleg rekenvoorbeeld.** De oudergroep heeft 14 keer geen stroke en 6 keer wel, dus $Gini = 1 - (14/20)^2 - (6/20)^2 = 1 - 0{,}49 - 0{,}09 = 0{,}42$. Bij de vraag `age` ≤ 60 komen links 11 mensen zonder stroke (Gini 0) en rechts 9 mensen, waarvan 6 met stroke (Gini 0,444). Gewogen is dat 11/20 · 0 + 9/20 · 0,444 = 0,20, dus de information gain is 0,42 − 0,20 = 0,22. Bij `hypertension` blijven allebei de groepen rommelig en is de gain bijna 0. De boom zou hier dus voor leeftijd kiezen. Met entropie krijg je dezelfde keuze.

Nu checken we of onze functies hetzelfde uitrekenen als scikit-learn. Een boom met `max_depth=1` (een *stump*) kiest precies één split, en dat is de beste eerste split op de hele trainingsset. We rekenen die na met onze eigen functies.

In [ ]:
# Prompt 3: Decision Tree stroke prediction
def compare_stump(stump, X, y):
    """Vergelijkt de eerste split van sklearn met de eigen functies."""
    tree_ = stump.tree_
    left, right = tree_.children_left[0], tree_.children_right[0]
    feature = X.columns[tree_.feature[0]]
    threshold = tree_.threshold[0]
    mask = (X[feature] <= threshold).to_numpy()
    y = np.asarray(y, dtype=int)
    # Information gain volgens sklearn, uit de opgeslagen impurities
    weights = tree_.weighted_n_node_samples
    sklearn_gain = tree_.impurity[0] - (
        weights[left] * tree_.impurity[left]
        + weights[right] * tree_.impurity[right]
    ) / weights[0]
    print(f"Beste eerste split volgens sklearn: {feature} <= {threshold:.2f}")
    comparison = pd.DataFrame(
        {
            "eigen functies": [
                gini(y), gini(y[mask]), gini(y[~mask]),
                information_gain(y, mask),
            ],
            "sklearn": [
                tree_.impurity[0], tree_.impurity[left],
                tree_.impurity[right], sklearn_gain,
            ],
        },
        index=["Gini root", "Gini links", "Gini rechts", "information gain"],
    )
    comparison["verschil"] = (
        comparison["eigen functies"] - comparison["sklearn"]
    ).abs()
    return comparison


# Zonder class_weight, zodat sklearn dezelfde ongewogen Gini berekent
stump = DecisionTreeClassifier(max_depth=1, random_state=RANDOM_STATE)
stump.fit(X_train, y_train)
root_gini = gini(y_train)
print(
    f"Gini root eigen functie: {root_gini:.4f}, "
    f"stump.tree_.impurity[0]: {stump.tree_.impurity[0]:.4f}, "
    f"gelijk: {np.isclose(root_gini, stump.tree_.impurity[0])}"
)
compare_stump(stump, X_train, y_train).round(4)

**Uitleg.** Op de hele trainingsset kiest scikit-learn `age` ≤ 73,5 als eerste vraag. De kolom 'verschil' is overal 0, dus onze eigen functies rekenen precies hetzelfde als scikit-learn. De information gain is klein (0,001), omdat de root al bijna zuiver is: 98,5% heeft geen stroke, dus de Gini is daar maar 0,03. De stump krijgt expres geen `class_weight`, anders rekent scikit-learn met een gewogen Gini en klopt de vergelijking niet meer.

#### 5.6.3 Pruning en de relatie met under- en overfitting
Als je een boom niet tegenhoudt, blijft hij doorsplitsen tot elk blad zuiver is. Dan leert hij de trainingsdata gewoon uit zijn hoofd: **overfitting**. Pruning is snoeien, en dat kan op twee manieren:

- **Pre-pruning:** vooraf stoppen, bijvoorbeeld met een maximale diepte (`max_depth`) of een minimaal aantal mensen per blad (`min_samples_leaf`).
- **Post-pruning:** de boom eerst helemaal laten groeien en daarna takken wegknippen die weinig toevoegen. Scikit-learn doet dat met *cost-complexity pruning*:

$$R_\alpha(T) = R(T) + \alpha \cdot |T|$$

Hier is $R(T)$ hoe slecht de boom de trainingsdata past (de onzuiverheid van de bladeren) en $|T|$ het aantal bladeren. Elk blad kost dus een soort boete $\alpha$ (`ccp_alpha`). Hoe hoger $\alpha$, hoe meer er geknipt wordt. Knip je te veel, dan wordt de boom te simpel en krijg je **underfitting**.

We laten dit zien met grafieken van de train- en validatie-F1. Dat doen we met cross-validation op de trainingsset, zodat de validatieset schoon blijft voor de echte test in 5.6.4.

(Breiman et al., 1984; scikit-learn developers, z.d.-e)

In [ ]:
# Prompt 3: Decision Tree stroke prediction
MAX_DEPTHS = np.arange(1, 21)

balanced_tree = DecisionTreeClassifier(
    class_weight="balanced", random_state=RANDOM_STATE
)


def plot_validation_curve(estimator, X, y, param_name, param_range,
                          log_scale=False):
    """Tekent de train- en validatie-F1 voor een reeks parameterwaarden."""
    cv = StratifiedKFold(
        n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_STATE
    )
    train_scores, val_scores = validation_curve(
        estimator, X, y, param_name=param_name, param_range=param_range,
        cv=cv, scoring=F1_BEST_SCORER, n_jobs=N_JOBS,
    )
    scores = pd.DataFrame(
        {
            "train F1": train_scores.mean(axis=1),
            "validatie F1": val_scores.mean(axis=1),
            "validatie std": val_scores.std(axis=1),
        },
        index=pd.Index(param_range, name=param_name),
    )
    best_value = scores["validatie F1"].idxmax()

    fig, ax = plt.subplots(figsize=(9, 5))
    ax.plot(param_range, scores["train F1"], marker="o", label="train")
    ax.plot(param_range, scores["validatie F1"], marker="o",
            label="validatie (CV-folds)")
    # Band van één standaardafwijking over de folds
    ax.fill_between(
        param_range,
        scores["validatie F1"] - scores["validatie std"],
        scores["validatie F1"] + scores["validatie std"],
        alpha=0.2,
    )
    ax.axvline(best_value, color="grey", linestyle="--",
               label=f"beste {param_name}: {best_value:.3g}")
    if log_scale:
        ax.set_xscale("log")
    ax.set_title(f"Train- en validatie-F1 per {param_name} (Decision Tree)")
    ax.set_xlabel(param_name)
    ax.set_ylabel("F1 bij beste drempel")
    ax.legend()
    plt.tight_layout()
    plt.show()
    return scores


depth_scores = plot_validation_curve(
    balanced_tree, X_train, y_train, "max_depth", MAX_DEPTHS
)
depth_scores.round(3)

**Uitleg grafiek (pre-pruning).** Bij diepte 1 en 2 zijn train en validatie allebei laag: de boom is te simpel, dus underfitting. Rond diepte 5 is de validatie-F1 het hoogst (0,149). Daarna gaat de train-F1 door tot 0,86 bij diepte 20, terwijl de validatie-F1 zakt naar ongeveer 0,08. De boom leert de trainingsdata dan uit zijn hoofd: overfitting. Hoe groter het gat tussen de twee lijnen, hoe meer overfitting.

Nu post-pruning. `cost_complexity_pruning_path` geeft alle waarden van $\alpha$ waarbij er een tak afvalt. Bij de grootste blijft alleen de root over. We testen 15 waarden die op een log-schaal gelijk verdeeld zijn.

In [ ]:
# Prompt 3: Decision Tree stroke prediction
MIN_ALPHA = 1e-5
N_ALPHAS = 15

pruning_path = balanced_tree.cost_complexity_pruning_path(X_train, y_train)
max_alpha = pruning_path.ccp_alphas.max()
print(
    f"Het pruning-pad heeft {len(pruning_path.ccp_alphas)} mogelijke "
    f"ccp_alpha's, van 0 tot {max_alpha:.4f}"
)
# Log-verdeeld, omdat de meeste alpha's heel klein zijn
alpha_range = np.geomspace(MIN_ALPHA, max_alpha, N_ALPHAS)
alpha_scores = plot_validation_curve(
    balanced_tree, X_train, y_train, "ccp_alpha", alpha_range,
    log_scale=True,
)
alpha_scores.round(4)

**Uitleg grafiek (post-pruning).** Deze grafiek lees je andersom. Links (alpha heel klein) wordt bijna niks geknipt: de train-F1 is 1,0 en de validatie-F1 maar 0,04, dus heel sterke overfitting. Naar rechts wordt de boom kleiner en stijgt de validatie-F1, tot het hoogste punt bij alpha ≈ 0,0013 (0,141). Helemaal rechts blijft alleen de root over en zijn train en validatie allebei laag: underfitting.

#### 5.6.4 Hyperparameter tuning
**Technieken tegen imbalance:** (1) `class_weight="balanced"`, waardoor fouten op stroke-patiënten zwaarder tellen; (2) de beslisdrempel afstemmen (zie 5.1). Standaardiseren is niet nodig: een boom vergelijkt steeds één feature met een grens, en schalen verandert de volgorde van de waarden niet.

| Hyperparameter | Waarden | Soort |
|---|---|---|
| `criterion` | gini, entropy | Splitscriterium |
| `max_depth` | 3, 5, 7, 10 | Pre-pruning, rond het optimum uit de grafiek |
| `min_samples_leaf` | 10, 50, 100 | Pre-pruning |
| `ccp_alpha` | 0; 0,0005; 0,002 | Post-pruning |

Dat zijn 2 × 4 × 3 × 3 = 72 combinaties, elk met 5-fold cross-validation.

In [ ]:
# Prompt 3: Decision Tree stroke prediction
def best_params_table(experiment, param_grid):
    """Zet de gekozen hyperparameters en de drempel in een tabel."""
    params = experiment.best_model.get_params()
    table = pd.Series(
        {name.split("__")[1]: params[name] for name in param_grid},
        dtype=object,
    )
    table["drempel"] = round(experiment.threshold, 3)
    return table.to_frame("gekozen waarde")


tree_pipeline = Pipeline([
    ("model", DecisionTreeClassifier(
        class_weight="balanced", random_state=RANDOM_STATE
    )),
])
tree_grid = {
    "model__criterion": ["gini", "entropy"],
    "model__max_depth": [3, 5, 7, 10],
    "model__min_samples_leaf": [10, 50, 100],
    "model__ccp_alpha": [0.0, 0.0005, 0.002],
}

tree = ModelExperiment(
    name="Decision Tree",
    pipeline=tree_pipeline,
    param_grid=tree_grid,
    techniques="class weights + drempel",
)
tree.run(data)
results.add(tree)
tree.plot_evaluation(y_val)
best_params_table(tree, tree_grid)

**Resultaat beslisboom.** De beste boom gebruikt Gini, diepte 5 en minstens 10 mensen per blad, zonder `ccp_alpha`. Dat klopt met de dieptegrafiek. Diepte 5 snoeit al genoeg, dus `ccp_alpha` voegt niks meer toe. De validatie-F1 is 0,088 (precision 0,059, recall 0,175), de laagste van alle modellen. Met de standaarddrempel 0,5 was het 0,076. De drempel is hoog (0,84), omdat de class weights alle kansen omhoog duwen. Bovendien heeft een boom met diepte 5 maar 27 bladeren, dus ook maar 27 verschillende kansen. Daardoor kun je de drempel niet fijn afstellen.

#### 5.6.5 Interpretatie: de boom en de belangrijkste features
Het voordeel van een beslisboom is dat je kunt zien waarom hij iets voorspelt. Hieronder staan de bovenste niveaus van de beste boom en daarna de feature importances.

In [ ]:
# Prompt 3: Decision Tree stroke prediction
CLASS_NAMES = ["geen stroke", "stroke"]
# Diepte 0 t/m 2 tekenen, dat zijn de bovenste 3 niveaus
PLOT_DEPTH = 2


def plot_top_levels(model, feature_names, depth):
    """Tekent de bovenste niveaus van een beslisboom."""
    fig, ax = plt.subplots(figsize=(22, 9))
    plot_tree(
        model, max_depth=depth, feature_names=feature_names,
        class_names=CLASS_NAMES, filled=True, proportion=True,
        rounded=True, fontsize=9, ax=ax,
    )
    ax.set_title(f"Bovenste {depth + 1} niveaus van de beste Decision Tree")
    plt.tight_layout()
    plt.show()


best_tree = tree.best_model.named_steps["model"]
plot_top_levels(best_tree, list(X_train.columns), PLOT_DEPTH)
print(f"Diepte van de beste boom: {best_tree.get_depth()}")
print(f"Aantal bladeren: {best_tree.get_n_leaves()}")

In [ ]:
# Prompt 3: Decision Tree stroke prediction
N_TOP_FEATURES = 10


def plot_feature_importances(model, feature_names, n_top):
    """Tekent de belangrijkste feature importances als staafgrafiek."""
    importances = (
        pd.Series(model.feature_importances_, index=feature_names)
        .sort_values(ascending=False)
        .head(n_top)
    )
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.barh(importances.index, importances.values)
    # Belangrijkste feature bovenaan
    ax.invert_yaxis()
    ax.set_title(f"Top {n_top} feature importances van de beste Decision Tree")
    ax.set_xlabel("Feature importance (aandeel van de totale impuritydaling)")
    ax.set_ylabel("Feature")
    plt.tight_layout()
    plt.show()
    return importances.round(3).to_frame("importance")


plot_feature_importances(best_tree, X_train.columns, N_TOP_FEATURES)

**Interpretatie.** Bijna elke vraag bovenin de boom gaat over leeftijd: de root is `age` ≤ 51,5. Hoe verder naar rechts (ouder), hoe blauwer, dus hoe meer risico. Leeftijd heeft een importance van 0,87, glucose 0,07 en bmi 0,03. De rest doet bijna niks. Feature importance zegt hoeveel een feature in totaal heeft geholpen om de Gini te verlagen, maar niet of een hogere waarde meer of minder risico geeft. Dat zie je alleen in de boom zelf. De root staat op [0,5; 0,5] omdat we class weights gebruiken. Daardoor zijn de kansen van dit model een risicoscore en geen echte kans.

(scikit-learn developers, z.d.-e)

### 5.7 Ensembles: Random Forest, Gradient Boosting en XGBoost

Code in deze paragraaf: [Claude, 2026. Prompt 4: Stroke prediction notebook models](https://claude.ai/share/741e7b67-5336-4153-965e-d840720b4f7d).

#### 5.7.1 Hoe werken deze ensembles?
Eén boom is wankel. Dus neem je er heel veel en combineer je ze. Als ze allemaal andere fouten maken, vallen die fouten tegen elkaar weg. Er zijn twee manieren om dat te doen.

**Bagging (bootstrap aggregation).** Je maakt nieuwe datasets door met teruglegging patiënten te trekken (*bootstrap*). Sommige mensen zitten er dan dubbel in en ongeveer 37% zit er helemaal niet in. Op elke dataset train je een boom, en aan het eind neem je het gemiddelde (*aggregation*). Die bomen hebben niks met elkaar te maken, dus je kunt ze tegelijk trainen: **parallel**. Het middelen zorgt dat het model minder heen en weer schiet (lagere variantie). **Random Forest** is bagging met één extra trucje: bij elke split mag een boom maar uit een deel van de features kiezen (`max_features`). Anders begint elke boom met leeftijd en lijken ze allemaal op elkaar.

**Boosting.** Hier train je de bomen na elkaar. Elke nieuwe boom kijkt waar het model tot nu toe nog fout zit (de residuen, het echte label min de voorspelde kans) en probeert dat te verbeteren. Elke boom telt maar een klein beetje mee, dat is de learning rate $\eta$:

$$F_m(x) = F_{m-1}(x) + \eta \cdot h_m(x)$$

Met een kleine $\eta$ heb je meer bomen nodig, maar overfit het model minder snel. Het lijkt op gradient descent: kleine stapjes de goede kant op. Omdat boom $m$ de fouten van boom $m-1$ nodig heeft, kan dit niet tegelijk: **sequentieel**. Boosting maakt het model vooral nauwkeuriger (lagere bias). `GradientBoostingClassifier` doet precies dit.

**XGBoost** is gradient boosting, maar dan opgevoerd. Er zit regularisatie in de doelfunctie, dus een boete op te veel bladeren ($\gamma$) en op te grote bladwaarden ($\lambda$), zodat hij minder overfit:

$$\text{Obj} = \sum_i L(y_i, \hat{y}_i) + \sum_m \Big(\gamma T_m + \tfrac{1}{2}\lambda \lVert w_m \rVert^2\Big)$$

Daarnaast zoekt XGBoost de beste split binnen een boom op meerdere kernen tegelijk, met histogrammen. De bomen komen nog steeds na elkaar, maar elke boom is veel sneller gebouwd.

| Ensemble | Techniek | Sequentieel of parallel? | Verwachte trainingstijd |
|---|---|---|---|
| Random Forest | Bagging | Parallel: bomen zijn onafhankelijk | Gemiddeld, sneller met meer kernen |
| Gradient Boosting | Boosting | Sequentieel | Veel |
| XGBoost | Boosting | Sequentieel tussen bomen, parallel binnen een boom | Weinig |

In 5.7.5 meten we of dit klopt.

(Breiman, 1996; Breiman, 2001; Friedman, 2001; Chen & Guestrin, 2016; De Haagse Hogeschool, 2026b)

#### 5.7.2 Random Forest
Voor alle drie de ensembles doen we dezelfde stappen. Daarom zetten we die eerst in een hulpfunctie.

**Technieken tegen imbalance:** (1) `class_weight="balanced_subsample"`: de gewichten worden per bootstrap-steekproef opnieuw berekend; (2) beslisdrempel afstemmen.

| Hyperparameter | Waarden | Betekenis |
|---|---|---|
| `n_estimators` | 300 (vast) | Aantal bomen; meer bomen geeft geen overfitting, alleen meer rekentijd |
| `max_depth` | 8, None | Maximale diepte per boom |
| `min_samples_leaf` | 10, 30 | Minimaal aantal patiënten per blad |
| `max_features` | sqrt, 0,5 | Hoeveel features elke split mag bekijken |

In [ ]:
# Prompt 4: Stroke prediction notebook models
def run_experiment(name, pipeline, param_grid, techniques):
    """Traint het model, zet het in results en toont de evaluatie."""
    experiment = ModelExperiment(name, pipeline, param_grid, techniques)
    experiment.run(data)
    results.add(experiment)
    experiment.plot_evaluation(y_val)
    return experiment

In [ ]:
# Prompt 4: Stroke prediction notebook models
# Random Forest met gebalanceerde gewichten per bootstrap-sample
forest_pipeline = Pipeline([
    ("model", RandomForestClassifier(
        n_estimators=300,
        class_weight="balanced_subsample",
        n_jobs=1,
        random_state=RANDOM_STATE,
    )),
])
forest_grid = {
    "model__max_depth": [8, None],
    "model__min_samples_leaf": [10, 30],
    "model__max_features": ["sqrt", 0.5],
}
forest = run_experiment(
    "Random Forest", forest_pipeline, forest_grid,
    techniques="class weights + drempel",
)

**Resultaat Random Forest.** Het beste waren bomen met maximaal diepte 8, minstens 30 mensen per blad en `sqrt` features per split. Dat is sterke regularisatie, en dat is logisch: met maar 414 stroke-patiënten leren diepe bomen die uit hun hoofd. In de cross-validation scoorde hij goed (CV-F1 ± 0,17), maar op de validatieset maar ± 0,09. Met de standaarddrempel 0,5 was het zelfs iets beter (± 0,10). De drempel die we op de trainingsdata kozen, werkt dus minder goed op nieuwe data. Eigenlijk is de drempel zelf overfit.

#### 5.7.3 Gradient Boosting
**Technieken tegen imbalance:** `GradientBoostingClassifier` heeft geen `class_weight`, dus we gebruiken (1) random undersampling in de pipeline en (2) de beslisdrempel. `subsample=0.8` betekent dat elke boom een willekeurige 80% van de rijen ziet (*stochastic gradient boosting*).

| Hyperparameter | Waarden | Betekenis |
|---|---|---|
| `sampling_strategy` | 0,1; 0,25 | Verhouding stroke : geen stroke na undersampling |
| `n_estimators` | 100, 300 | Aantal bomen |
| `learning_rate` | 0,03; 0,1 | Hoe zwaar elke nieuwe boom meetelt |
| `max_depth` | 2, 3 | Diepte per (zwakke) boom |

In [ ]:
# Prompt 4: Stroke prediction notebook models
# Eerst undersampling, daarna Gradient Boosting op de kleinere set
boosting_pipeline = Pipeline([
    ("sampler", RandomUnderSampler(random_state=RANDOM_STATE)),
    ("model", GradientBoostingClassifier(
        subsample=0.8,
        random_state=RANDOM_STATE,
    )),
])
boosting_grid = {
    "sampler__sampling_strategy": [0.1, 0.25],
    "model__n_estimators": [100, 300],
    "model__learning_rate": [0.03, 0.1],
    "model__max_depth": [2, 3],
}
boosting = run_experiment(
    "Gradient Boosting", boosting_pipeline, boosting_grid,
    techniques="undersampling + drempel",
)

**Resultaat Gradient Boosting.** Het beste was 100 bomen met diepte 2, learning rate 0,1 en undersampling tot 1 stroke op 10 gezonde mensen. Daardoor traint elk model op maar ongeveer 16% van de gezonde mensen (4.140 van de 26.421). Er gaat dus informatie verloren, maar het tunen gaat wel snel. Hele kleine bomen werkten het best. Dat past bij de data, want het risico komt vooral uit een paar losse dingen: leeftijd, glucose en bloeddruk. De validatie-F1 is 0,103 (precision 0,075, recall 0,165), met de standaarddrempel 0,093.

#### 5.7.4 XGBoost
**Technieken tegen imbalance:** (1) `scale_pos_weight`: het gewicht van de stroke-klasse. 64 is ongeveer de verhouding geen stroke : stroke, en 10 is een mildere weging; (2) beslisdrempel afstemmen. `subsample`, `colsample_bytree` en `min_child_weight` staan vast als extra regularisatie.

| Hyperparameter | Waarden | Betekenis |
|---|---|---|
| `n_estimators` | 200, 400 | Aantal bomen |
| `learning_rate` | 0,03; 0,1 | Gewicht van elke nieuwe boom |
| `max_depth` | 2, 3, 4 | Diepte per boom |
| `scale_pos_weight` | 10, 64 | Gewicht van de stroke-klasse |

In [ ]:
# Prompt 4: Stroke prediction notebook models
# XGBoost met extra gewicht voor de strokeklasse via scale_pos_weight
xgboost_pipeline = Pipeline([
    ("model", XGBClassifier(
        subsample=0.8,
        colsample_bytree=0.8,
        min_child_weight=5,
        tree_method="hist",
        eval_metric="logloss",
        n_jobs=1,
        random_state=RANDOM_STATE,
    )),
])
xgboost_grid = {
    "model__n_estimators": [200, 400],
    "model__learning_rate": [0.03, 0.1],
    "model__max_depth": [2, 3, 4],
    "model__scale_pos_weight": [10, 64],
}
xgboost = run_experiment(
    "XGBoost", xgboost_pipeline, xgboost_grid,
    techniques="scale_pos_weight + drempel",
)

**Resultaat XGBoost.** Het beste was 200 bomen, diepte 3, learning rate 0,03 en `scale_pos_weight` 10. Gek genoeg werkt 10 beter dan 64, terwijl 64 de 'eerlijke' verhouding is. Maar we stellen de drempel daarna toch nog af, dus de klassen hoeven niet helemaal in balans. Met 64 roept hij waarschijnlijk te snel 'stroke', en dan krijg je veel vals alarm. De validatie-F1 is 0,118 (precision 0,076, recall 0,262), de beste van de drie ensembles.

#### 5.7.5 Trainingstijd: sequentieel tegenover parallel
Om eerlijk te vergelijken, krijgen alle drie de ensembles hetzelfde aantal bomen (300) en dezelfde diepte (3). Random Forest en XGBoost meten we op 1 kern en op alle kernen. Gradient Boosting kan alleen op 1 kern. Per model nemen we de mediaan van 3 runs, zodat één toevallig trage run niet telt.

In [ ]:
# Prompt 4: Stroke prediction notebook models
# Gelijke instellingen voor een eerlijke vergelijking
N_TREES = 300
TREE_DEPTH = 3
N_TIMING_RUNS = 3


def make_timing_models():
    """Maakt de modellen voor het tijdsexperiment met gelijke instellingen."""
    shared = {
        "n_estimators": N_TREES,
        "max_depth": TREE_DEPTH,
        "random_state": RANDOM_STATE,
    }
    return {
        "Random Forest (1 kern)": RandomForestClassifier(
            n_jobs=1, **shared),
        "Random Forest (alle kernen)": RandomForestClassifier(
            n_jobs=N_JOBS, **shared),
        "Gradient Boosting (1 kern)": GradientBoostingClassifier(**shared),
        "XGBoost (1 kern)": XGBClassifier(
            tree_method="hist", n_jobs=1, **shared),
        "XGBoost (alle kernen)": XGBClassifier(
            tree_method="hist", n_jobs=N_JOBS, **shared),
    }


def time_fit(model, X, y):
    """Geeft de mediane trainingstijd in seconden over meerdere runs."""
    durations = []
    for _ in range(N_TIMING_RUNS):
        # Elke run een nieuw, ongetraind model
        fresh_model = clone(model)
        start = time.perf_counter()
        fresh_model.fit(X, y)
        durations.append(time.perf_counter() - start)
    return np.median(durations)


def measure_training_times(models, X, y):
    """Meet per model de trainingstijd en zet de resultaten in een tabel."""
    rows = [
        {"Model": name, "Trainingstijd (s)": time_fit(model, X, y)}
        for name, model in models.items()
    ]
    return (
        pd.DataFrame(rows)
        .sort_values("Trainingstijd (s)")
        .reset_index(drop=True)
        .round(2)
    )


def plot_training_times(training_times):
    """Tekent de trainingstijden als horizontale staafgrafiek."""
    fig, ax = plt.subplots(figsize=(8, 4))
    bars = ax.barh(
        training_times["Model"], training_times["Trainingstijd (s)"]
    )
    ax.bar_label(bars, fmt="%.2f s", padding=3)
    ax.set_title(f"Trainingstijd ({N_TREES} bomen, diepte {TREE_DEPTH})")
    ax.set_xlabel("Mediane trainingstijd (seconden)")
    ax.set_ylabel("Model")
    plt.tight_layout()
    plt.show()

In [ ]:
# Prompt 4: Stroke prediction notebook models
training_times = measure_training_times(
    make_timing_models(), X_train, y_train
)
plot_training_times(training_times)
training_times

**Interpretatie.** De meting bevestigt de tabel uit 5.7.1:
- **Gradient Boosting is het traagst.** Alle 300 bomen worden na elkaar gebouwd, op één kern.
- **Random Forest** wordt sneller als het meerdere kernen mag gebruiken, omdat de bomen los van elkaar zijn. Met meer kernen wordt het verschil groter.
- **XGBoost is het snelst**, zelfs op één kern. Het is net zo sequentieel als Gradient Boosting, maar door de histogrammen kost elke boom veel minder rekenwerk.

De precieze seconden verschillen per computer, maar de volgorde niet.

(Chen & Guestrin, 2016; scikit-learn developers, z.d.-f)

### 5.8 Zelf samengesteld ensemble

Code in deze paragraaf: [Claude, 2026. Prompt 5: Stroke prediction ensemble notebook](https://claude.ai/share/2c71f5e0-e6bd-45fb-bc00-eb4b77ae5a51).

#### 5.8.1 Hoe werkt een voting-ensemble?
Je neemt een paar verschillende modellen en laat ze samen stemmen. Bij **hard voting** zegt elk model 0 of 1 en wint de meerderheid. Bij **soft voting** geeft elk model een kans en neem je het (gewogen) gemiddelde:

$$\hat{p}_{ensemble} = \frac{\sum_m w_m \, \hat{p}_m}{\sum_m w_m}$$

Wij kiezen **soft voting**, om twee redenen. Een model dat heel zeker is (90%) telt dan zwaarder dan een model dat twijfelt (51%). En je krijgt een kans, dus je kunt de drempel nog afstellen. Bij hard voting krijg je alleen 0 of 1, en dan werkt onze belangrijkste techniek tegen imbalance niet meer.

#### 5.8.2 Keuze van de modellen
Combineren heeft alleen zin als de modellen verschillende fouten maken. Daarom pakken we drie soorten modellen:
- **Logistic Regression:** een lineair model en het beste losse model;
- **Random Forest:** bagging;
- **XGBoost:** boosting.

Alle drie geven ze een kans. De SVM geeft geen kans, en KNN en de losse boom scoorden te laag. Gradient Boosting laten we weg, omdat het hetzelfde doet als XGBoost, alleen langzamer.

Om te checken of de modellen echt verschillen, kijken we hoe sterk hun out-of-fold scores samenhangen. We gebruiken de Spearman-correlatie, omdat die alleen naar de volgorde kijkt. Dat is nodig, want de SVM geeft een afstand en de andere modellen een kans.

(Kuncheva, 2014; scikit-learn developers, z.d.-f)

In [ ]:
# Prompt 5: Stroke prediction ensemble notebook
experiments = [log_reg, knn, svm, tree, forest, boosting, xgboost]

# Zet de out-of-fold scores van alle modellen naast elkaar
oof_df = pd.DataFrame(
    {experiment.name: experiment.oof_scores for experiment in experiments}
)

# Spearman kijkt naar de rangorde, dus SVM-afstanden zijn ook bruikbaar
correlation = oof_df.corr(method="spearman")

fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(
    correlation,
    annot=True,
    fmt=".2f",
    cmap="viridis",
    vmin=0,
    vmax=1,
    square=True,
    ax=ax,
)
ax.set_title("Spearman-correlatie tussen out-of-fold scores per model")
ax.set_xlabel("Model")
ax.set_ylabel("Model")
plt.tight_layout()
plt.show()

**Interpretatie.** Bij de sterke modellen (LR, SVM, Random Forest, Gradient Boosting en XGBoost) is de correlatie 0,96 tot 0,99. Ze zetten de patiënten dus bijna in dezelfde volgorde van risico, omdat ze allemaal vooral naar leeftijd kijken. Alleen de losse boom (± 0,8) en KNN (± 0,9) wijken meer af, maar die scoren zelf te laag. Je kunt dus al verwachten dat combineren niet heel veel gaat opleveren. We kiezen toch LR, Random Forest en XGBoost: het zijn drie verschillende families, en een gemiddelde van drie modellen is meestal stabieler dan één model.

#### 5.8.3 Hyperparameter tuning
De drie modellen houden hun beste instellingen van daarvoor, en ook hun eigen techniek tegen imbalance. We tunen alleen hoe zwaar elk model meetelt. Daarna stellen we net als bij de andere modellen de drempel af.

| Gewichten (LR, RF, XGB) | Idee |
|---|---|
| [1, 1, 1] | Alle modellen even zwaar |
| [2, 1, 1] | Logistic Regression dubbel (beste losse model) |
| [1, 1, 2] | XGBoost dubbel (sterkste boommodel) |

In [ ]:
# Prompt 5: Stroke prediction ensemble notebook
# Soft voting middelt de kansen van drie verschillende soorten modellen
voting_model = VotingClassifier(
    estimators=[
        ("logreg", clone(log_reg.best_model)),
        ("forest", clone(forest.best_model)),
        ("xgboost", clone(xgboost.best_model)),
    ],
    voting="soft",
)

# Alleen de gewichten tunen, de losse modellen zijn al getuned
ENSEMBLE_PARAM_GRID = {
    "weights": [[1, 1, 1], [2, 1, 1], [1, 1, 2]],
}

ensemble = ModelExperiment(
    name="Eigen ensemble",
    pipeline=voting_model,
    param_grid=ENSEMBLE_PARAM_GRID,
    techniques="technieken losse modellen + drempel",
)
ensemble.run(data)
results.add(ensemble)
ensemble.plot_evaluation(y_val)

**Resultaat eigen ensemble.** Het beste was LR dubbel ([2, 1, 1]), en dat is logisch, want LR was los het beste model. Het ensemble haalt een CV-F1 van ± 0,17 en een validatie-F1 van ± 0,12 (precision ± 0,07, recall ± 0,32). Dat is beter dan Random Forest en ongeveer gelijk aan XGBoost, maar LR alleen (0,131) is nog net iets beter. Precies wat de heatmap al liet zien: de modellen maken grotendeels dezelfde fouten, dus combineren levert weinig op. Wel vindt het ensemble bijna een derde van de stroke-patiënten, meer dan XGBoost los (0,26).

(Kuncheva, 2014)